In [ ]:
# Cell 1: Install Dependencies
!pip install requests beautifulsoup4 trafilatura openai tldextract urllib3
!pip install lxml html2text

In [ ]:
# Cell 2: Import Libraries
import requests
from bs4 import BeautifulSoup
import trafilatura
import json
import re
from urllib.parse import urljoin, urlparse
import time
import random
from typing import Dict, List, Optional, Set
import tldextract
from collections import defaultdict
import html2text

In [ ]:
# Cell 3: Core Scraping Engine
class SmartCompanyScraper:
    """Intelligent scraper with multiple fallback strategies"""
    
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
            'Accept-Encoding': 'gzip, deflate',
            'DNT': '1',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
        })
        self.delay = random.uniform(1, 2)
        self.relevant_keywords = {
            'about', 'contact', 'services', 'products', 'team', 'company',
            'about-us', 'contact-us', 'our-services', 'what-we-do',
            'solutions', 'portfolio', 'clients', 'testimonials'
        }
        
    def _extract_domain_info(self, url: str) -> Dict:
        """Extract domain components"""
        extracted = tldextract.extract(url)
        return {
            'domain': f"{extracted.domain}.{extracted.suffix}",
            'subdomain': extracted.subdomain,
            'base_url': f"https://{extracted.domain}.{extracted.suffix}"
        }
    
    def _find_relevant_pages(self, base_url: str, soup: BeautifulSoup) -> List[str]:
        """Intelligently find relevant pages using fuzzy matching"""
        relevant_urls = set()
        
        # Method 1: Check sitemap
        sitemap_urls = self._check_sitemap(base_url)
        if sitemap_urls:
            relevant_urls.update(
                url for url in sitemap_urls 
                if any(keyword in url.lower() for keyword in self.relevant_keywords)
            )
        
        # Method 2: Extract navigation links
        for link in soup.find_all('a', href=True):
            href = link['href']
            text = link.get_text(strip=True).lower()
            
            # Skip non-relevant links
            if any(skip in text for skip in ['login', 'signup', 'privacy', 'terms', 'cookie']):
                continue
            
            # Check if link text contains relevant keywords
            if any(keyword in text for keyword in self.relevant_keywords):
                full_url = urljoin(base_url, href)
                if self._is_valid_url(full_url, base_url):
                    relevant_urls.add(full_url)
        
        # Always include homepage and common pages
        for page in ['/about', '/contact', '/services', '/about-us', '/contact-us']:
            relevant_urls.add(urljoin(base_url, page))
        
        return list(relevant_urls)[:8]  # Limit to 8 most relevant pages
    
    def _check_sitemap(self, base_url: str) -> List[str]:
        """Try to fetch and parse sitemap"""
        sitemap_urls = [f"{base_url}/sitemap.xml", f"{base_url}/sitemap_index.xml"]
        
        for sitemap_url in sitemap_urls:
            try:
                response = self.session.get(sitemap_url, timeout=10)
                if response.status_code == 200:
                    soup = BeautifulSoup(response.content, 'xml')
                    urls = [loc.text for loc in soup.find_all('loc')]
                    if urls:
                        return urls
            except:
                continue
        return []
    
    def _is_valid_url(self, url: str, base_url: str) -> bool:
        """Validate URL is from same domain and not a file"""
        try:
            parsed = urlparse(url)
            base_parsed = urlparse(base_url)
            
            # Same domain check
            if parsed.netloc and parsed.netloc != base_parsed.netloc:
                return False
            
            # Skip files and anchors
            skip_extensions = ['.pdf', '.jpg', '.png', '.doc', '.xls', '.zip', '#']
            if any(url.lower().endswith(ext) for ext in skip_extensions):
                return False
            
            return True
        except:
            return False
    
    def _clean_html_content(self, html: str) -> str:
        """Aggressively clean HTML to minimize tokens"""
        soup = BeautifulSoup(html, 'lxml')
        
        # Remove unwanted elements
        unwanted_tags = [
            'script', 'style', 'nav', 'footer', 'header', 
            'aside', 'iframe', 'noscript', 'form', 'button',
            'svg', 'canvas', 'video', 'audio', 'embed'
        ]
        for tag in unwanted_tags:
            for element in soup.find_all(tag):
                element.decompose()
        
        # Remove common non-content classes/IDs
        unwanted_classes = [
            'cookie', 'banner', 'popup', 'modal', 'advertisement',
            'social', 'share', 'sidebar', 'widget', 'menu', 'navigation',
            'footer', 'header', 'newsletter', 'subscribe'
        ]
        for class_name in unwanted_classes:
            for element in soup.find_all(class_=re.compile(class_name, re.I)):
                element.decompose()
        
        # Convert to clean text
        h = html2text.HTML2Text()
        h.ignore_links = False
        h.ignore_images = True
        h.ignore_emphasis = False
        h.body_width = 0
        
        text = h.handle(str(soup))
        
        # Clean up whitespace
        text = re.sub(r'\n\s*\n', '\n\n', text)
        text = re.sub(r' +', ' ', text)
        
        return text.strip()
    
    def _extract_contact_info(self, text: str) -> Dict:
        """Extract emails and phone numbers with validation"""
        contact = {
            'emails': [],
            'phone_numbers': []
        }
        
        # Email extraction with validation
        email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
        found_emails = re.findall(email_pattern, text)
        
        # Filter out common false positives
        invalid_patterns = ['example', 'test', 'user', 'admin', 'your', 'email', '.png', '.jpg']
        valid_emails = [
            email for email in found_emails 
            if not any(pattern in email.lower() for pattern in invalid_patterns)
        ]
        contact['emails'] = list(dict.fromkeys(valid_emails))[:3]  # Remove duplicates, limit to 3
        
        # Phone number extraction with international formats
        phone_patterns = [
            r'\+\d{1,3}[\s\-]?\d{1,4}[\s\-]?\d{1,4}[\s\-]?\d{1,4}',  # International
            r'\d{3}[\s\-]?\d{3}[\s\-]?\d{4}',  # US/Standard
            r'\(\d{3}\)\s?\d{3}[\s\-]?\d{4}',  # (123) 456-7890
            r'\d{2,4}[\s\-]\d{3,4}[\s\-]\d{3,4}',  # Variable length
        ]
        
        phone_numbers = set()
        for pattern in phone_patterns:
            matches = re.findall(pattern, text)
            phone_numbers.update(matches)
        
        # Filter invalid numbers
        valid_phones = [
            num for num in phone_numbers 
            if len(re.sub(r'\D', '', num)) >= 7
        ]
        contact['phone_numbers'] = list(valid_phones)[:3]
        
        return contact
    
    def scrape_company(self, url: str) -> Dict:
        """Main scraping orchestration with fallbacks"""
        time.sleep(self.delay)  # Rate limiting
        
        result = {
            'url': url,
            'success': False,
            'pages_scraped': 0,
            'content': '',
            'metadata': {},
            'error': None
        }
        
        try:
            # Strategy 1: Try trafilatura for main content
            downloaded = trafilatura.fetch_url(url)
            if downloaded:
                content = trafilatura.extract(
                    downloaded,
                    include_comments=False,
                    include_tables=False,
                    no_fallback=False,
                    favor_precision=True
                )
                if content and len(content.strip()) > 100:
                    result['success'] = True
                    result['content'] = self._clean_html_content(downloaded)
                    result['method'] = 'trafilatura'
                    
                    # Extract metadata
                    soup = BeautifulSoup(downloaded, 'lxml')
                    result['metadata'] = {
                        'title': soup.title.string if soup.title else '',
                        'description': soup.find('meta', attrs={'name': 'description'})['content'] if soup.find('meta', attrs={'name': 'description'}) else ''
                    }
                    return result
            
            # Strategy 2: Multi-page intelligent scraping
            response = self.session.get(url, timeout=15, verify=False)
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'lxml')
                
                # Find relevant pages
                relevant_pages = self._find_relevant_pages(
                    self._extract_domain_info(url)['base_url'], 
                    soup
                )
                
                # Scrape relevant pages
                all_content = []
                pages_successful = 0
                
                for page_url in relevant_pages[:5]:  # Limit to 5 pages max
                    try:
                        time.sleep(random.uniform(0.5, 1))
                        page_response = self.session.get(page_url, timeout=10, verify=False)
                        if page_response.status_code == 200:
                            cleaned = self._clean_html_content(page_response.content.decode())
                            if len(cleaned) > 200:
                                all_content.append(f"--- {page_url} ---\n{cleaned}")
                                pages_successful += 1
                    except:
                        continue
                
                if all_content:
                    result['success'] = True
                    result['content'] = '\n\n'.join(all_content)
                    result['pages_scraped'] = pages_successful
                    result['method'] = 'multi_page'
                    
                    # Metadata from main page
                    result['metadata'] = {
                        'title': soup.title.string if soup.title else '',
                        'description': soup.find('meta', attrs={'name': 'description'})['content'] if soup.find('meta', attrs={'name': 'description'}) else ''
                    }
                    return result
            
            # Strategy 3: Simple request fallback
            response = self.session.get(url, timeout=15, verify=False)
            if response.status_code == 200:
                cleaned = self._clean_html_content(response.content.decode())
                result['success'] = True
                result['content'] = cleaned
                result['method'] = 'simple'
                return result
            
        except Exception as e:
            result['error'] = str(e)
        
        return result

In [ ]:
# Cell 4: AI Analysis Engine
class CompanyAnalyzer:
    """AI-powered company analysis with anti-hallucination measures"""
    
    def __init__(self, openai_api_key: str):
        import openai
        self.client = openai.OpenAI(api_key=openai_api_key)
        self.model = "gpt-3.5-turbo-1106"  # Good balance of speed/cost/context
        
    def _chunk_content(self, content: str, max_chars: int = 3000) -> List[str]:
        """Smart chunking to fit within token limits"""
        if len(content) <= max_chars:
            return [content]
        
        chunks = []
        paragraphs = content.split('\n\n')
        current_chunk = ""
        
        for para in paragraphs:
            if len(current_chunk) + len(para) < max_chars:
                current_chunk += para + '\n\n'
            else:
                chunks.append(current_chunk.strip())
                current_chunk = para + '\n\n'
        
        if current_chunk:
            chunks.append(current_chunk.strip())
        
        return chunks
    
    def analyze_company(self, scraped_data: Dict) -> Dict:
        """Generate company profile with strict anti-hallucination"""
        
        if not scraped_data.get('success'):
            return self._empty_profile(scraped_data.get('url', ''))
        
        # Prepare content chunks
        content_chunks = self._chunk_content(scraped_data['content'])
        metadata = scraped_data.get('metadata', {})
        
        # Extract contact info separately (no AI needed)
        contact_info = SmartCompanyScraper()._extract_contact_info(scraped_data['content'])
        
        # Prepare prompt with anti-hallucination measures
        system_prompt = """You are a precise company data extractor. Follow these rules STRICTLY:
1. ONLY extract information that is EXPLICITLY mentioned in the text
2. If information is NOT found, use "N/A" or empty strings
3. NEVER fabricate, guess, or assume any data
4. For emails and phone numbers, ONLY use what's provided
5. For company name, use the official name found in the title/header
6. Be conservative - it's better to say "N/A" than to guess
7. Return ONLY the JSON object, no other text"""

        user_prompt = f"""Based ONLY on this company website content, extract the following information.
If ANY field is not explicitly mentioned, use "N/A".

Metadata:
- Page Title: {metadata.get('title', 'N/A')}
- Description: {metadata.get('description', 'N/A')}

Content (first part):
{content_chunks[0][:4000]}

Return this EXACT JSON structure:
{{
  "company_name": "Official company name from website",
  "website": "{scraped_data.get('url', '')}",
  "description": "Brief factual description from the website",
  "industry": "Industry based on content",
  "services": ["Service 1 mentioned", "Service 2 mentioned"],
  "products": ["Product 1 mentioned"],
  "about": "Brief about section content",
  "emails": {json.dumps(contact_info['emails'])},
  "phone_numbers": {json.dumps(contact_info['phone_numbers'])},
  "address": "Physical address if mentioned, else N/A",
  "social_media": {{}},
  "founded": "Year if mentioned, else N/A",
  "team_size": "Number if mentioned, else N/A"
}}"""

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.1,  # Very low temperature to reduce hallucination
                max_tokens=1000,
                response_format={"type": "json_object"}
            )
            
            result = json.loads(response.choices[0].message.content)
            
            # Ensure all required fields exist
            return self._validate_and_fill(result, scraped_data.get('url', ''), contact_info)
            
        except Exception as e:
            return self._fallback_extraction(scraped_data, contact_info)
    
    def _validate_and_fill(self, data: Dict, url: str, contact: Dict) -> Dict:
        """Ensure all required fields exist with proper types"""
        required_structure = {
            'company_name': '',
            'website': url,
            'description': '',
            'industry': 'N/A',
            'services': [],
            'products': [],
            'about': 'N/A',
            'emails': contact['emails'],
            'phone_numbers': contact['phone_numbers'],
            'address': 'N/A',
            'social_media': {},
            'founded': 'N/A',
            'team_size': 'N/A'
        }
        
        for key, default in required_structure.items():
            if key not in data or data[key] is None:
                data[key] = default
        
        # Ensure arrays stay as arrays
        for array_field in ['services', 'products', 'emails', 'phone_numbers']:
            if not isinstance(data.get(array_field), list):
                data[array_field] = [data[array_field]] if data.get(array_field) else []
        
        return data
    
    def _fallback_extraction(self, scraped_data: Dict, contact: Dict) -> Dict:
        """Basic extraction without AI when API fails"""
        content = scraped_data.get('content', '')
        metadata = scraped_data.get('metadata', {})
        
        # Extract company name from title
        company_name = metadata.get('title', '')
        if '|' in company_name:
            company_name = company_name.split('|')[0].strip()
        elif '-' in company_name:
            company_name = company_name.split('-')[0].strip()
        
        return {
            'company_name': company_name,
            'website': scraped_data.get('url', ''),
            'description': metadata.get('description', 'N/A'),
            'industry': 'N/A',
            'services': [],
            'products': [],
            'about': 'N/A',
            'emails': contact['emails'],
            'phone_numbers': contact['phone_numbers'],
            'address': 'N/A',
            'social_media': {},
            'founded': 'N/A',
            'team_size': 'N/A'
        }
    
    def _empty_profile(self, url: str) -> Dict:
        """Empty profile for failed scrapes"""
        return {
            'company_name': 'N/A',
            'website': url,
            'description': 'Failed to scrape',
            'industry': 'N/A',
            'services': [],
            'products': [],
            'about': 'N/A',
            'emails': [],
            'phone_numbers': [],
            'address': 'N/A',
            'social_media': {},
            'founded': 'N/A',
            'team_size': 'N/A'
        }

In [ ]:
# Cell 5: Main Pipeline
def process_companies(urls: List[str], openai_api_key: str) -> List[Dict]:
    """
    Main function to process multiple company URLs
    DO NOT modify this function signature
    """
    scraper = SmartCompanyScraper()
    analyzer = CompanyAnalyzer(openai_api_key)
    
    results = []
    
    for i, url in enumerate(urls):
        print(f"Processing {i+1}/{len(urls)}: {url}")
        
        # Step 1: Scrape
        scraped_data = scraper.scrape_company(url)
        print(f"  Scraped: {scraped_data['success']} (Method: {scraped_data.get('method', 'N/A')})")
        
        # Step 2: Analyze
        profile = analyzer.analyze_company(scraped_data)
        results.append(profile)
        
        # Rate limiting between companies
        if i < len(urls) - 1:
            time.sleep(random.uniform(1, 2))
    
    return results

In [ ]:
# Cell 6: Input and Execution
# This cell will prompt for input and process URLs
from google.colab import output

# Get OpenAI API Key
api_key = input("Enter your OpenAI API key: ")

# Get URLs
print("\nEnter company URLs (one per line, press Enter twice to finish):")
urls = []
while True:
    url = input()
    if url == "":
        break
    urls.append(url.strip())

print(f"\nProcessing {len(urls)} companies...\n")

# Process companies
results = process_companies(urls, api_key)

# Save results
with open('results.json', 'w') as f:
    json.dump(results, f, indent=2)

# Display results
print("\n" + "="*50)
print("RESULTS")
print("="*50)
print(json.dumps(results, indent=2))

# Download link
from google.colab import files
files.download('results.json')

print("\n✅ Processing complete! Results saved to results.json")